In [3]:
import torch
from torch.utils.data import DataLoader, Dataset
import polars as pl
import cv2
from torchvision.transforms import v2

class PlayerCropDataset(Dataset):
    def __init__(self, df: pl.DataFrame, transform=None):
        self.crop_paths = df["crop_path"].to_list()
        team_map = {
            'left' : 0,
            'right' : 1
        }
        self.labels = [team_map[t] for t in df["team"].to_list()]
        self.transform = transform
        
    def __len__(self):
        return len(self.crop_paths)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        
        bgr = cv2.imread(self.crop_paths[idx])
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            image_tensor = self.transform(rgb)
        else:
            # Permute shifts (H, W, C) to the PyTorch format (C, H, W) and normalize to 255 pixel vals
            image_tensor = (torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0)
        
        label_tensor = torch.tensor(self.labels[idx], dtype=torch.long)
        return image_tensor, label_tensor

In [4]:
import polars as pl
from torchvision.transforms import v2
train_transforms = v2.Compose(
    [
        v2.ToImage(),  # Converts uint8 numpy (H, W, C) -> torch tensor (C, H, W)
        v2.Resize((128, 64)),
        v2.RandomHorizontalFlip(p=0.5),  # Augmentation for generalization
        v2.ToDtype(torch.float32, scale=True),  # Scales [0, 255] -> [0.0, 1.0]
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

val_transforms = v2.Compose(
    [
        v2.ToImage(),
        v2.Resize((128, 64)),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

path = "data/processed/crop_manifest.parquet"
manifest_df = pl.read_parquet(path)

train_df = manifest_df.filter(pl.col("split") == "train")
val_df = manifest_df.filter(pl.col("split") == "val")


train_dataset = PlayerCropDataset(train_df, train_transforms)
val_dataset = PlayerCropDataset(val_df, val_transforms)

train_loader = DataLoader(train_dataset, batch_size = 64, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 64, shuffle = False)

images, labels = next(iter(train_loader))

print(f"Batch images shape: {images.shape}")  # Expecting: [64, 3, 128, 64], (B, C, H, W)
print(f"Batch labels shape: {labels.shape}")  # Expecting: [64], (B)
print(f"Label sample: {labels[:5]}")  # Expecting: tensor([1, 0, 1, 0, 0]...)

Batch images shape: torch.Size([64, 3, 128, 64])
Batch labels shape: torch.Size([64])
Label sample: tensor([1, 0, 0, 1, 0])
